# Sentinel-2 drought indices

This notebook is the executable companion to the Quarto tutorial. It discovers public Sentinel-2 imagery, masks poor-quality pixels, calculates NDVI, NDMI, and NDWI, and derives a simple anomaly and VCI time series.

**Internet access is required** for the STAC search and raster download.

## Setup

Install the packages once with `pip install -r requirements.txt` from the repository root.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import geopandas as gpd
from shapely.geometry import box as make_box
from pystac_client import Client
import planetary_computer
from odc.stac import load

## Find Sentinel-2 scenes

The AOI is a small example area near Salzburg. Coordinates are `(west, south, east, north)` in EPSG:4326.

In [ ]:
bbox = (13.00, 47.70, 13.10, 47.76)
aoi = gpd.GeoDataFrame(
    {"name": ["example AOI"]},
    geometry=[make_box(*bbox)],
    crs="EPSG:4326",
)

catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2024-04-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 30}},
)
items = list(search.items())
if not items:
    raise RuntimeError("No Sentinel-2 scenes matched the AOI and date filters.")
print(f"Found {len(items)} scenes")

## Load bands and mask clouds

Reflectance is scaled by 10,000 in Sentinel-2 L2A assets. The SCL band supplies a per-pixel quality classification.

In [ ]:
cube = load(
    items,
    bands=["B03", "B04", "B08", "B11", "SCL"],
    bbox=bbox,
    resolution=20,
    crs="EPSG:32633",
    groupby="solar_day",
    chunks={},
)

reflectance = cube[["B03", "B04", "B08", "B11"]] / 10_000
scl = cube["SCL"]
clear = scl.isin([4, 5, 6, 11])
masked = reflectance.where(clear)
masked

## Calculate spectral indices

NDVI responds mainly to greenness, NDMI to vegetation water content, and NDWI to open water and surface wetness.

In [ ]:
def normalised_difference(a, b):
    denominator = a + b
    return xr.where(denominator != 0, (a - b) / denominator, np.nan)

indices = xr.Dataset({
    "NDVI": normalised_difference(masked.B08, masked.B04),
    "NDMI": normalised_difference(masked.B08, masked.B11),
    "NDWI": normalised_difference(masked.B03, masked.B08),
})
indices

In [ ]:
indices.NDVI.isel(time=0).plot(cmap="YlGn", vmin=0, vmax=1, figsize=(7, 4))
plt.title("NDVI for the first available observation")
plt.tight_layout()

## Aggregate to indicators

A spatial median gives one value per observation. The baseline below is only the sample itself; operational monitoring needs several comparable years.

In [ ]:
series = indices.median(dim=("y", "x"), skipna=True).sortby("time")
baseline = series.NDVI.median("time")
ndvi_anomaly = series.NDVI - baseline

ndvi_min = series.NDVI.min("time")
ndvi_max = series.NDVI.max("time")
vci = xr.where(
    ndvi_max != ndvi_min,
    100 * (series.NDVI - ndvi_min) / (ndvi_max - ndvi_min),
    np.nan,
).rename("VCI")

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ndvi_anomaly.plot(ax=axes[0], marker="o", color="firebrick")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("NDVI anomaly")
vci.plot(ax=axes[1], marker="o", color="darkgreen")
axes[1].set_ylim(0, 100)
axes[1].set_title("Vegetation Condition Index (VCI)")
plt.tight_layout()

## Interpretation

Low NDVI or VCI can indicate vegetation stress, but can also result from harvest, planting, pests, soil background, or residual clouds. Sentinel-2 has no thermal band, so TCI and VHI require a complementary thermal product such as MODIS or Landsat.